In [14]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install scipy

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------------ --------------------------- 11.0/36.5 MB 63.1 MB/s eta 0:00:01
   --------------------------- ------------ 25.4/36.5 MB 68.0 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 62.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import h5py
import numpy as np
import os
import torch
from scipy.signal import butter, sosfiltfilt

# -----------------------------
# Paths and preprocessing settings
# -----------------------------

base_dir = r"C:/Users/Micah/utah-neuro/MATLAB_Jupyter/00_data_preparation/aligned_h5"

kdf_path = os.path.join(base_dir, "TrainingData_20260723-153348 (2)_kdf.h5")
ns5_training_path = os.path.join(base_dir, "TrainingData_20260723-153348 (2)_ns5_aligned.h5")
output_path = os.path.join(base_dir, "Aligned_Train_Data_Preprocessed (2).pt")

FS_HZ = 30000.0
NOTCH_BAND_HZ = (59.0, 61.0)
LOWPASS_HZ = 1000.0
HIGHPASS_HZ = 40.0
FILTER_ORDER = 4
STD_EPS = 1e-8

# -----------------------------
# EMG preprocessing
# -----------------------------

def preprocess_emg(ns5_samples):
    """
    Preprocess continuous EMG [samples, channels].

    Order:
      1. 59-61 Hz Butterworth band-stop
      2. 1000 Hz Butterworth low-pass
      3. 40 Hz Butterworth high-pass
      4. Per-channel centering and scaling to standard deviation 1

    Channels are filtered separately to keep peak memory manageable.
    sosfiltfilt applies zero-phase filtering, avoiding phase shifts.
    """
    x = np.asarray(ns5_samples)

    if x.ndim != 2:
        raise ValueError(f"Expected a 2D [samples, channels] array, got {x.shape}")

    nyquist = FS_HZ / 2.0
    if not (0 < NOTCH_BAND_HZ[0] < NOTCH_BAND_HZ[1] < nyquist):
        raise ValueError("Invalid notch band for the configured sample rate.")
    if not (0 < HIGHPASS_HZ < LOWPASS_HZ < nyquist):
        raise ValueError("Expected 0 < high-pass < low-pass < Nyquist.")

    notch_sos = butter(
        FILTER_ORDER,
        [NOTCH_BAND_HZ[0] / nyquist, NOTCH_BAND_HZ[1] / nyquist],
        btype="bandstop",
        output="sos",
    )
    lowpass_sos = butter(
        FILTER_ORDER,
        LOWPASS_HZ / nyquist,
        btype="lowpass",
        output="sos",
    )
    highpass_sos = butter(
        FILTER_ORDER,
        HIGHPASS_HZ / nyquist,
        btype="highpass",
        output="sos",
    )

    # Preallocate float32 output; scipy uses higher precision internally.
    processed = np.empty(x.shape, dtype=np.float32)

    for channel in range(x.shape[1]):
        signal = np.asarray(x[:, channel], dtype=np.float64)

        signal = sosfiltfilt(notch_sos, signal)
        signal = sosfiltfilt(lowpass_sos, signal)
        signal = sosfiltfilt(highpass_sos, signal)

        channel_mean = float(np.mean(signal))
        channel_std = float(np.std(signal))

        if not np.isfinite(channel_std) or channel_std < STD_EPS:
            raise ValueError(
                f"Channel {channel} has invalid/near-zero standard deviation "
                f"after filtering: {channel_std}"
            )

        processed[:, channel] = (
            (signal - channel_mean) / channel_std
        ).astype(np.float32)

    return processed

# -----------------------------
# Load KDF training data
# -----------------------------

with h5py.File(kdf_path, "r") as kdf:
    train_niptime = kdf["trainNIPtime"][:].flatten()
    train_kinematics = kdf["trainKin"][:]

# Keep only first 7 kinematic values
train_kinematics = train_kinematics[:, :7]

# Sort by trainNIPtime
sort_idx = np.argsort(train_niptime)
train_niptime_sorted = train_niptime[sort_idx]
train_kinematics_sorted = train_kinematics[sort_idx]

# -----------------------------
# Load and preprocess full NS5 training data
# -----------------------------

with h5py.File(ns5_training_path, "r") as f:
    print("NS5 datasets:", list(f.keys()))
    ns5_training = f["data"][:]

# Make sure shape is samples x channels
if ns5_training.shape[0] == 32:
    ns5_training = ns5_training.T

print("Raw ns5_training shape:", ns5_training.shape)
num_ns5_samples = ns5_training.shape[0]

print(
    "Preprocessing EMG: 59-61 Hz notch, 1000 Hz low-pass, "
    "40 Hz high-pass, per-channel SD=1"
)
ns5_training = preprocess_emg(ns5_training)

# Verify normalization before alignment.
channel_means = np.mean(ns5_training, axis=0)
channel_stds = np.std(ns5_training, axis=0)
print("Maximum absolute channel mean:", float(np.max(np.abs(channel_means))))
print("Channel standard-deviation range:", float(channel_stds.min()), "to", float(channel_stds.max()))

# -----------------------------
# Convert trainNIPtime to row index inside ns5_training
# -----------------------------

train_niptime_start = train_niptime_sorted[0]

kdf_indices_0indexed = (
    train_niptime_sorted - train_niptime_start
).astype(np.int64)

if kdf_indices_0indexed.min() < 0:
    raise ValueError("Some KDF indices are below 0.")

if kdf_indices_0indexed.max() >= num_ns5_samples:
    raise ValueError(
        f"Some KDF indices exceed ns5_training length. "
        f"Max requested index: {kdf_indices_0indexed.max()}, "
        f"NS5 samples available: {num_ns5_samples}"
    )

# -----------------------------
# Fill every NS5 sample with preceding KDF trainKin
# -----------------------------

all_ns5_indices = np.arange(num_ns5_samples)

preceding_kdf_pos = np.searchsorted(
    kdf_indices_0indexed,
    all_ns5_indices,
    side="right"
) - 1

valid_mask = preceding_kdf_pos >= 0

all_ns5_indices = all_ns5_indices[valid_mask]
preceding_kdf_pos = preceding_kdf_pos[valid_mask]

filled_train_kinematics = train_kinematics_sorted[preceding_kdf_pos]
filled_train_niptime = train_niptime_start + all_ns5_indices
filled_ns5_vectors = ns5_training[all_ns5_indices, :]
ns5_samples_1indexed = all_ns5_indices + 1

# -----------------------------
# Convert matched data to PyTorch tensors
# -----------------------------

aligned_tensor = {
    "ns5_sample": torch.tensor(ns5_samples_1indexed, dtype=torch.long),
    "trainNIPtime": torch.tensor(filled_train_niptime, dtype=torch.long),
    "ns5_vector": torch.tensor(filled_ns5_vectors, dtype=torch.float32),
    "trainKin": torch.tensor(filled_train_kinematics, dtype=torch.float32),
    "preprocessing": {
        "sample_rate_hz": FS_HZ,
        "notch_band_hz": NOTCH_BAND_HZ,
        "lowpass_hz": LOWPASS_HZ,
        "highpass_hz": HIGHPASS_HZ,
        "filter_order": FILTER_ORDER,
        "filter_method": "Butterworth SOS, zero-phase sosfiltfilt",
        "normalization": "per-channel mean centering and population standard deviation 1",
    },
}

# -----------------------------
# One-hot encode trainKin
# -----------------------------

def argmax_one_hot_keep_zeros(x):
    max_indices = torch.argmax(x, dim=1)
    one_hot = torch.zeros_like(x)

    nonzero_rows = x.sum(dim=1) != 0
    rows = torch.arange(x.shape[0], device=x.device)

    one_hot[rows[nonzero_rows], max_indices[nonzero_rows]] = 1
    return one_hot

aligned_tensor["trainKin"] = argmax_one_hot_keep_zeros(aligned_tensor["trainKin"])

# -----------------------------
# Save tensor file
# -----------------------------

torch.save(aligned_tensor, output_path)

# -----------------------------
# Print checks
# -----------------------------

print("Number of full NS5 samples used:", len(aligned_tensor["trainNIPtime"]))
print("ns5_vector shape:", aligned_tensor["ns5_vector"].shape)
print("trainKin shape:", aligned_tensor["trainKin"].shape)
print("Preprocessing metadata:", aligned_tensor["preprocessing"])
print("\nSaved preprocessed tensor file to:")
print(output_path)


NS5 datasets: ['data']
Raw ns5_training shape: (7231081, 32)
Preprocessing EMG: 59-61 Hz notch, 1000 Hz low-pass, 40 Hz high-pass, per-channel SD=1
Maximum absolute channel mean: 1.4139953652403392e-09
Channel standard-deviation range: 0.9906351566314697 to 0.9976381063461304
Number of full NS5 samples used: 7231081
ns5_vector shape: torch.Size([7231081, 32])
trainKin shape: torch.Size([7231081, 7])
Preprocessing metadata: {'sample_rate_hz': 30000.0, 'notch_band_hz': (59.0, 61.0), 'lowpass_hz': 1000.0, 'highpass_hz': 40.0, 'filter_order': 4, 'filter_method': 'Butterworth SOS, zero-phase sosfiltfilt', 'normalization': 'per-channel mean centering and population standard deviation 1'}

Saved preprocessed tensor file to:
C:/Users/Micah/utah-neuro/MATLAB_Jupyter/00_data_preparation/aligned_h5\Aligned_Train_Data_Preprocessed (2).pt
